# Model Finetuning
We will finetune the model in the following steps:
- Take in the finetuned dataset
- Convert to the proper format (tensors)
- Then finetune the model based

Some imports for libaries we will use and helper classes for finetuning the model. Instead of writing our own training loop and evaluation function, we will be using the given functions from pytorch directly. 

In [1]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
import json # For our manually labeled finetune train data
from PIL import Image
import torchvision.transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset
import os
import sys

base = "https://raw.githubusercontent.com/pytorch/vision/main/references/detection"
files = ["engine.py", "utils.py", "coco_utils.py", "coco_eval.py", "transforms.py"]

for f in files:
    os.system(f"curl -o finetune_helpers/{f} {base}/{f}")

sys.path.append("./finetune_helpers")
from finetune_helpers.engine import train_one_epoch
from finetune_helpers import utils

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4063  100  4063    0     0  20494      0 --:--:-- --:--:-- --:--:-- 20417
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8388  100  8388    0     0  45813      0 --:--:-- --:--:-- --:--:-- 45836
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8409  100  8409    0     0  44421      0 --:--:-- --:--:-- --:--:-- 44492
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6447  100  6447    0     0  31260      0 --:--:-- --:--:-- --:--:-- 31296
  % Total    % Received % Xferd  Average Speed   Tim

# Steps 1 & 2
First we will take in the finetuned dataset (*annotations.json*) and convert it into a list of dictionaries where each element of the dictinoary is either "boxes", i.e. the location of boxes on our classified images, or "labels", i.e. 1 since we are classifying tree (1) and not tree (0). 
We do this in the __init__ function.

Since we will be working with the data when training/finetuning we create a class that inherits from Dataset. 

In [ ]:
class TreeDataset(Dataset):
    def __init__(self, annotations_path):
        self.root_path = annotations_path
        with open(self.root_path + "annotations.json") as f:
            all_annotations = json.load(f)

        self.annotations = []
        for a in all_annotations:
            if (all_annotations[a] != []): # drop pictures with no trees
                self.annotations.append((a, all_annotations[a]))
        self.transform = T.ToTensor()

    def __len__(self):
        return len(self.annotations)  # how many images total

    def __getitem__(self, idx):
        # print(self.annotations)
        # print(self.annotations[1])
        image_name, boxes = self.annotations[idx]
        img = Image.open(self.root_path + "raw_images/" + image_name).convert("RGB")
        img_tensor = self.transform(img)
        
        target = {
            "boxes":  torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.ones(len(boxes), dtype=torch.int64),
            "image_id": torch.tensor([idx])
        }
        
        return img_tensor, target

# Sanity Check:
dataset_full = TreeDataset("./Training Classifier/")
print(f"Total images: {len(dataset_full)}")

Total images: 256


# Step 3
Now we are modifying the pre-trained Faster R-CNN model.  

In particular, we are modifying the classification layer of our Faster R-CNN model. Instead of multiple classes like airplanes, persons, chairs, etc., we will now have **only 2 classes**: tree (1) or no tree (0)

This code is heavily inspired by the [TorchVision Object Detection Finetuning Tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html#torchvision-object-detection-finetuning-tutorial)

In [3]:
def get_fine_tune_ready_model():
    # load a model pre-trained on COCO
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

    num_classes = 2  # 1 class (tree) + background

    # get number of input features for the classifier (from the conv. layers / pooling)
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Predicted trees that overlap more than 30% are the same tree
    model.roi_heads.nms_thresh = 0.3
    
    # Freezing most parameters (everything but the final head/FC layer)
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.rpn.parameters():
        param.requires_grad = False
    
    return model

# Step 4
Now we will finetune the model. This code is almost directly taken from the [TorchVision Object Detection Finetuning Tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html#torchvision-object-detection-finetuning-tutorial) 

In [4]:
# train on the accelerator (mac) or on the CPU, if an accelerator is not available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# our dataset has two classes only - background and tree
num_classes = 2
dataset = TreeDataset('./Training Classifier/')

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=utils.collate_fn
)

# get the model using our helper function
model = get_fine_tune_ready_model()

# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=8,
    gamma=0.3
)

num_epochs = 24

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()

print("That's it!")

/Users/paulrabel/tree_detector/model/finetune_helpers/engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [ 0/64]  eta: 0:04:17  lr: 0.000084  loss: 5.2129 (5.2129)  loss_classifier: 0.8704 (0.8704)  loss_box_reg: 0.2802 (0.2802)  loss_objectness: 3.8865 (3.8865)  loss_rpn_box_reg: 0.1759 (0.1759)  time: 4.0216  data: 0.0876
Epoch: [0]  [10/64]  eta: 0:01:19  lr: 0.000877  loss: 4.8549 (5.0319)  loss_classifier: 0.5652 (0.5929)  loss_box_reg: 0.2354 (0.2305)  loss_objectness: 3.8865 (3.9980)  loss_rpn_box_reg: 0.1850 (0.2105)  time: 1.4707  data: 0.0666
Epoch: [0]  [20/64]  eta: 0:00:58  lr: 0.001670  loss: 4.7247 (4.8327)  loss_classifier: 0.5486 (0.5642)  loss_box_reg: 0.2107 (0.2205)  loss_objectness: 3.8306 (3.8505)  loss_rpn_box_reg: 0.1791 (0.1975)  time: 1.1875  data: 0.0616
Epoch: [0]  [30/64]  eta: 0:00:42  lr: 0.002463  loss: 4.5145 (4.6878)  loss_classifier: 0.3666 (0.4701)  loss_box_reg: 0.2027 (0.2166)  loss_objectness: 3.7239 (3.8059)  loss_rpn_box_reg: 0.1760 (0.1952)  time: 1.1503  data: 0.0585
Epoch: [0]  [40/64]  eta: 0:00:29  lr: 0.003256  loss: 4.7464 (4.704

# Final Step:
Save the learned weights

In [5]:
torch.save(model.state_dict(), "weights.pt")

# More Adjustments

Currently, we only finetuned the classification layer of our model (the very last layer). Using the following parameters:
- `batch size = 4`
- `learning rate = 0.005`
- `momentum = 0.9`
- `weight decay = 0.0005`
- `step size = 8`
- `gamma = 0.3`
- `epochs = 24`

the classification loss was reduced from $~0.87$ to $~0.08$. 

However, the total loss stayed relatively constant, between $3$ and $5$, mainly due to the **loss_objectness** which consistenly is around $3.5$. **loss_objectness** is measuring the performance of our model determining whether any object is at a location. Since the model was trained on `COCO`, and satellite-view trees are not part of that training data, we must also update weights in the RPN layer of the model. Thus, we proceed with *progressive unfreezing*.

In [6]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

num_classes = 2
dataset = TreeDataset('./Training Classifier/')

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=utils.collate_fn
)

# get the model
model = get_fine_tune_ready_model()
model.load_state_dict(torch.load("weights.pt", weights_only=True))
for param in model.rpn.parameters(): # go deeper than before
    param.requires_grad = True
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.0005,
    momentum=0.9,
    weight_decay=0.0005
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=8,
    gamma=0.5
)

num_epochs = 12

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()

print("That's it!")

Epoch: [0]  [ 0/64]  eta: 0:02:26  lr: 0.000008  loss: 5.0491 (5.0491)  loss_classifier: 0.0695 (0.0695)  loss_box_reg: 0.1017 (0.1017)  loss_objectness: 4.6559 (4.6559)  loss_rpn_box_reg: 0.2221 (0.2221)  time: 2.2914  data: 0.0885
Epoch: [0]  [10/64]  eta: 0:01:19  lr: 0.000088  loss: 4.3758 (4.1253)  loss_classifier: 0.0799 (0.0782)  loss_box_reg: 0.1350 (0.1371)  loss_objectness: 3.8367 (3.7208)  loss_rpn_box_reg: 0.1872 (0.1891)  time: 1.4711  data: 0.0724
Epoch: [0]  [20/64]  eta: 0:01:03  lr: 0.000167  loss: 3.9165 (3.7846)  loss_classifier: 0.0888 (0.1072)  loss_box_reg: 0.1596 (0.1685)  loss_objectness: 3.2210 (3.3252)  loss_rpn_box_reg: 0.1764 (0.1837)  time: 1.3970  data: 0.0679
Epoch: [0]  [30/64]  eta: 0:00:48  lr: 0.000246  loss: 2.7264 (3.3158)  loss_classifier: 0.2624 (0.2236)  loss_box_reg: 0.3030 (0.2486)  loss_objectness: 1.8244 (2.6527)  loss_rpn_box_reg: 0.1737 (0.1910)  time: 1.3871  data: 0.0634
Epoch: [0]  [40/64]  eta: 0:00:33  lr: 0.000326  loss: 2.1724 (3.089

# Final Step

Now that the **loss_objectness** has decreased (from ~$4$ to now being ~$0.25$) and our total **loss** has also significantly decreased, we can save the final weights.

In [7]:
torch.save(model.state_dict(), "final_weights.pt")